In [25]:
# pip install mediapipe opencv-python

In [1]:
import cv2
import numpy as np
import mediapipe as mp
# import argparse
import os
# from datetime import datetime
from scipy.signal import find_peaks
import time
import math
import csv
from ultralytics import YOLO
# from IPython.display import display, clear_output
# from PIL import Image

In [2]:
# requirements = [
#     "opencv-python",
#     "mediapipe",
#     "numpy"
# ]

# with open("requirements.txt", "w") as f:
#     for r in requirements:
#         f.write(r + "\n")

# print("requirements.txt created!")

In [3]:
mpDraw = mp.solutions.drawing_utils
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence = 0.6)

# Drawing style helpers (optional customizations)
DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(255,0,0), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)

In [4]:
def partial_lap(lms, partial_lap_count, partial_state, left_line_x, right_line_x):

    central_point = (lms[23].x+lms[24].x)/2
    middle = (left_line_x + right_line_x)/2
    if central_point <= middle:
        if partial_state != "left":      # only count when arriving from right
            partial_lap_count += 1
            partial_state = "left"

    else:
        if partial_state != "right":     # only count when arriving from right
            partial_lap_count += 1
            partial_state = "right"
    return partial_lap_count, partial_state

In [5]:
# ---------- Count Laps ----------
def count_laps(lms, lap_count, state, left_line_x, right_line_x):
    
    # foot_positions = [lms[29].x, lms[31].x, lms[30].x, lms[32].x]
    #  lms[32].x, lms[17].x, lms[19].x, lms[18].x, lms[20].x      inculde hand touch
    max_foot_x = max(lms[29].x, lms[31].x, lms[30].x, lms[32].x, lms[17].x, lms[18].x)
    min_foot_x = min(lms[29].x, lms[31].x, lms[30].x, lms[32].x, lms[17].x, lms[18].x)
    
    # for foot_x in foot_positions:

    # --- check left line touch ---
    if min_foot_x <= left_line_x:
        if state != "left":      # only count when arriving from middle/right
            lap_count += 1
            state = "left"

    # --- check right line touch ---
    elif max_foot_x >= right_line_x:
        if state != "right":    # only count once per arrival
            lap_count += 1
            state = "right"

    # --- in between lines ---
    else:
        state = "middle"

    return lap_count, state

In [6]:
# TURNING ANGLE FUNCTION
def torso_angle(lms):
    l_sh, r_sh, l_hip, r_hip = lms[11], lms[12], lms[23], lms[24]
    sh_vec = (r_sh.x - l_sh.x, r_sh.y - l_sh.y)
    hip_vec = (r_hip.x - l_hip.x, r_hip.y - l_hip.y)
    
    dot = sh_vec[0]*hip_vec[0] + sh_vec[1]*hip_vec[1]
    mag1 = math.sqrt(sh_vec[0]**2 + sh_vec[1]**2)
    mag2 = math.sqrt(hip_vec[0]**2 + hip_vec[1]**2)
    
    if mag1 == 0 or mag2 == 0:
        return 0
    
    angle = math.degrees(math.acos(dot / (mag1 * mag2)))
    return angle


In [7]:
def turning(lms, frame_idx, fps, left_line_x, right_line_x, in_turn, turn_start_time, turn_angles_this_turn, turning_times, turn_angles):
    # Foot position for detecting turns
    foot_x = min(lms[29].x, lms[30].x)
    current_time = frame_idx / fps

    # ENTER TURN ZONE
    if not in_turn and (foot_x <= left_line_x or foot_x >= right_line_x):
        in_turn = True
        turn_start_time = current_time

    # RECORD ANGLES WHILE TURNING
    if in_turn:
        angle = torso_angle(lms)
        turn_angles_this_turn.append(angle)

    # EXIT TURN ZONE
    if in_turn and (left_line_x < foot_x < right_line_x):
        in_turn = False
        turn_end_time = current_time
        turn_duration = turn_end_time - turn_start_time
        turning_times.append(turn_duration)

        # Save average turn angle for this turn
        if len(turn_angles_this_turn) > 0:
            turn_angles.append(sum(turn_angles_this_turn) / len(turn_angles_this_turn))
            turn_angles_this_turn.clear()
    return in_turn, turn_start_time, turn_angles_this_turn, turning_times, turn_angles

    

In [39]:
def shuttle_time_score(time_sec, age, gender):
    """
    time_sec : float (time in seconds)
    age      : float
    gender   : 'boy' or 'girl'
    """

    gender = gender.lower()

    if 14 <= age <= 16:
        if gender == "boy":
            limits = [13, 13.8, 14.6, 15.4, 16.2]
        elif gender == "girl":
            limits = [13.5, 14.3, 15.1, 15.9, 16.7]
        else:
            return None

    elif 16 < age <= 18:
        if gender == "boy":
            limits = [12.8, 13.6, 14.4, 15.2, 16.0]
        elif gender == "girl":
            limits = [13.8, 14.6, 15.4, 16.2, 17.0]
        else:
            return None
    else:
        return None

    # Assign score
    if time_sec < limits[0]:
        return 6
    elif time_sec < limits[1]:
        return 5
    elif time_sec < limits[2]:
        return 4
    elif time_sec < limits[3]:
        return 3
    elif time_sec < limits[4]:
        return 2
    else:
        return 1


In [40]:
def line_touch_accuracy_score(correct_touches):
    if correct_touches >= 6:
        return 6
    elif correct_touches >= 5:
        return 5
    elif correct_touches >= 4:
        return 4
    elif correct_touches >= 3:
        return 3
    elif correct_touches >= 2:
        return 2
    else:
        return 1


In [41]:
# TURNING EFFICIENCY FUNCTION
def turning_efficiency_score(avg_turn_angle, avg_turn_time):
    
    if avg_turn_angle is None or avg_turn_time is None:
        return 1
    
    if avg_turn_angle < 20 and avg_turn_time < 0.25:
        return 6
    elif avg_turn_angle < 25:
        return 5
    elif avg_turn_angle < 35:
        return 4
    elif avg_turn_angle < 45:
        return 3
    elif avg_turn_angle < 60 or avg_turn_time < 0.6:
        return 2
    else:
        return 1


In [42]:
def compute_movement_metrics(hip_x_list, hip_y_list, time_list):
    # Convert to arrays
    hip_x_arr = np.array(hip_x_list)
    hip_y_arr = np.array(hip_y_list)
    t_arr = np.array(time_list)

    # -------- SPEED VAR (Stride timing consistency) --------
    inverted_signal = -hip_y_arr
    peaks, _ = find_peaks(inverted_signal, distance=5)

    if len(peaks) > 1:
        stride_times = t_arr[peaks]
        stride_intervals = np.diff(stride_times)
        speed_var = np.std(stride_intervals)
    else:
        speed_var = 0.2  # fallback for low stride count

    # -------- SWAY (Side-to-side control) --------
    sway = np.std(hip_x_arr)

    # -------- ACCEL VAR (Acceleration smoothness) --------
    if len(t_arr) > 2:
        velocities = np.diff(hip_y_arr) / np.diff(t_arr)
        accel = np.diff(velocities)
        accel_var = np.std(accel) if len(accel) > 1 else 1.0
    else:
        accel_var = 1.0

    return  round(speed_var, 3), round(sway, 3),  round(accel_var, 3)


In [43]:
def movement_control_score(speed_var, sway, accel_var):
    if speed_var < 0.05 and sway < 0.02 and accel_var < 0.04:
        return 6
    elif speed_var < 0.07:
        return 5
    elif speed_var < 0.09:
        return 4
    elif sway < 0.05:
        return 3
    elif sway < 0.08:
        return 2
    else:
        return 1


In [44]:
def predict_category(_score, lap_time_score):
    
    # Safety checks
    if _score < 0: _score = 0
    if _score > 20: _score = 20
    if lap_time_score < 0: lap_time_score = 0
    if lap_time_score > 5: lap_time_score = 5

    # -------------------------------
    # Top Tier Category (Excellent)
    # -------------------------------
    if _score >= 17:
        if lap_time_score >= 4:
            return "Excellent"
        else:
            return "Above Average"

    # -------------------------------
    # High-Mid Tier (Above Average / Average)
    # -------------------------------
    elif 14 <= _score <= 17:
        if lap_time_score >= 3:
            return "Above Average"
        elif lap_time_score == 2:
            return "Average"
        else:
            return "Below Average"

    # -------------------------------
    # Mid Tier (Average / Below Average)
    # -------------------------------
    elif 10 <= _score <= 13:
        if lap_time_score >= 3:
            return "Average"
        elif lap_time_score == 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lower Tier (Below Average)
    # -------------------------------
    elif 6 <= _score <= 9:
        if lap_time_score >= 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lowest Tier (Poor)
    # -------------------------------
    else:  # _score 0–5
        return "Poor"

In [45]:
def add_data(row):
    # row must be a list: ["value1", "value2", ...]
    with open("shuttle_run_results.csv", "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(row)


In [46]:
# open("shuttle_run_results.csv", "w").close()
# data = ["ID", "Name", "Age", "Gender", "Video_path", "Accurate_lap_count", "Partial_lap_count", "Avg_turn_angle", "Movement:speed_var", "Movement:sway", "Movement:accel_var", "Avg_turn_time", "Score_18", "Category"]
# add_data(data)

In [47]:
def threshold_line(path):
    
    model = YOLO("best.pt")
    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    # print("Left cone X:", left_avg)
    # print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    return left_avg, right_avg


In [48]:
def shuttle_run(ID, name, path, age, gender):
    
    # cap = cv2.VideoCapture(0)  # 0 = default camera
    # path = "img-5610-0q6jn2tf_fZX3eQZg.mov"
    cap = cv2.VideoCapture(path)
    

    # Define video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # remove in live case
    fps = cap.get(cv2.CAP_PROP_FPS)                            # frames per second
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))       # total frames
    if fps > 0:
        video_time = frame_count / fps

    # print("FPS:", fps)
    # print("Total Frames:", frame_count)
    # print("Video Duration (seconds):", video_time_seconds)


    # save the output video
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # out = cv2.VideoWriter(video_filename, fourcc, fps, (frame_width, frame_height))

    #threshold line calculate  
    left_line_x, right_line_x = 0.10, 0.85
    left_line_x, right_line_x = threshold_line(path)
    left_line_x = round(((left_line_x)/frame_width), 2) + 0.05
    right_line_x = round(((right_line_x)/frame_width), 2) - 0.05
    # print(f"{left_line_x} --- {right_line_x}")
    
    # total_time for 6 lap
    partial_lap_count = 0
    partial_state = "left"
    # partial_state = "right"
    time_seconds = frame_count/fps
    
    # accurate lap count const
    lap_count = 0
    state = "left"
    # state = "right"

    # turning const 
    frame_idx = 0
    turning_times = []
    turn_angles = []
    in_turn = False
    turn_start_time = 0
    turn_angles_this_turn = []

    # movement const
    hip_x_list = []
    hip_y_list = []
    time_list = []

    
    # start_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_idx += 1
        
        # if lap_count == 6 or partial_lap_count > 6:
        #     time_seconds = frame_idx/fps
        #     break

        # Convert BGR → RGB for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

         # Run pose detection
        results = pose.process(rgb)

        if results.pose_landmarks:
            # Enumerate all landmarks
            # for id, lm in enumerate(results.pose_landmarks.landmark):
            #     # Optional: Draw skeleton
            #     mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
                
            #     # Convert normalized landmark to pixel coordinates
            #     h, w, c = frame.shape
            #     cx, cy = int(lm.x * w), int(lm.y * h)
            #     # IDs to highlight: 23, 24, 25, 26
            #     if id in [23, 24, 25, 26]:
            #         # Draw circle on frame
            #         cv2.circle(frame, (cx, cy), 8, (0, 255, 0), -1)

            mpDraw.draw_landmarks(frame, results.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
            lms = results.pose_landmarks.landmark

            h, w, c = frame.shape
            cx1 = int(left_line_x * w)
            cx2 = int(right_line_x * w)
            cy1 = int(0.5 * h)
            cy2 = int(0.5 * h)
            cv2.circle(frame, (cx1, cy1), 8, (0, 255, 0), -1)
            cv2.circle(frame, (cx2, cy2), 8, (0, 255, 0), -1)
            cx = int((cx1+cx2)/2)
            cy = int((cy1+cy2)/2)
            cv2.circle(frame, (cx, cy), 8, (0, 0, 255), -1)
            
            
            # A. partial_lap_count_time
            partial_lap_count, partial_state = partial_lap(lms, partial_lap_count, partial_state, left_line_x, right_line_x)
            cv2.putText(frame, f"partial_lap_Count: {partial_lap_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
    
            # B. accurate_lap_count
            lap_count, state = count_laps(lms, lap_count, state, left_line_x, right_line_x)
            cv2.putText(frame, f"lap_Count: {lap_count}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)

            # condition for exact time note  

            # C. turning
            in_turn, turn_start_time, turn_angles_this_turn, turning_times, turn_angles = turning(lms, frame_idx, fps, left_line_x, right_line_x, in_turn, turn_start_time, turn_angles_this_turn, turning_times, turn_angles)
            
            # D. movement parameter
            hip_x = (lms[23].x + lms[24].x) / 2
            hip_y = (lms[23].y + lms[24].y) / 2

            hip_x_list.append(hip_x)
            hip_y_list.append(hip_y)
            time_list.append(frame_idx/fps)
         

        # out.write(frame)
        # cv2.imshow("Press 'q' to stop early", frame)
        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        # Show frame in Jupyter
        # clear_output(wait=True)
        # display(Image.fromarray(frame))

    cap.release()
    # out.release()
    cv2.destroyAllWindows()

    # print(f"Calculated time:--> {time.time()-start_time}")
    # A. lap-time score
    lap_time_score = shuttle_time_score(time_seconds, age, gender)
    print(time_seconds)
    
    # B. line touching score
    line_touch_score = line_touch_accuracy_score(lap_count)
    
    # C. turning score
    avg_turn_time = round(np.mean(turning_times), 2) if len(turning_times) else None
    avg_turn_angle = round(np.mean(turn_angles), 2) if len(turn_angles) else None
    turning_score = turning_efficiency_score(avg_turn_angle, avg_turn_time)
        
    # D. movement score
    speed_var, sway, accel_var = compute_movement_metrics(hip_x_list, hip_y_list, time_list)
    movement_score = movement_control_score(speed_var, sway, accel_var)

    # total_score_18
    _score = line_touch_score + turning_score + movement_score

    # predict category
    category = predict_category(_score, lap_time_score)
    

    # show data
    # if (data_print =='Y' or data_print == 'y'):
    #     print("printing data here------------->>")
    #     print(f"Toatl time:--> {time_seconds:.2f}")
    #     print(f"Accurate Lap Count:--> {lap_count} and partial Lap count:--> {partial_lap_count}")
    #     print(f"Avg turning angle:--> {avg_turn_angle:.2f} and Avg turning time:--> {avg_turn_time:.2f}")
    #     print(f"Movement--> (speed_var, sway, accel_var): ({speed_var:.3f}, {sway:.3f}, {accel_var:.3f})")
        

    # adding data
    data = [ID, name, age, gender, path, lap_count, partial_lap_count, avg_turn_angle, speed_var, sway, accel_var, avg_turn_time, _score, category]
    add_data(data)

    # print(step_times)
    # print("------------------------------------")
    # print(step_times_interwal)
    print(_score, "--", category)
    return _score, category
        

In [36]:
# ID = input("(DCXXXXX)Enter unique ID:")
# name = input("Name of Candidate:")
# path = input("Path of your Video:")
# # "demo-shuttle-run_HwmEdnDL.mp4"
# data_print = input("Wants to print data Y/N:")
# shuttle_score, category = shuttle_run(ID, name, path, data_print)
# print("printing result here-------------->>")
# print(f"Your High Knee Jump Score is--> {shuttle_score:.2f}/18 and Category--> {category}")

In [49]:

ID = "DCXXXX"
name = "Life"
path = "20251223_154042.mp4"
# path = "IMG_5805 (online-video-cutter.com).mp4"
age = 15
gender = "boy"  # gender : 'boy' or 'girl'
shuttle_score, category = shuttle_run(ID, name, path, age, gender)


14.661111111111111
12 -- Average


In [38]:
# path = "20251223_154122.mp4"
# left_line_x, right_line_x = threshold_line(path)
# print(f"{left_line_x} --- {right_line_x}")

In [46]:
# !pip uninstall opencv-python-headless -y
# !pip install opencv-python


In [47]:
# !pip install ultralytics opencv-python


In [48]:
# from ultralytics import YOLO
# import cv2

# # Load a pretrained YOLO model
# model = YOLO("yolov8n.pt")   # small, fast model

In [49]:
# results = model.track(source = "data/5 meter Shuttle (online-video-cutter.com).mp4", save = True)

In [50]:
# for box in results[4].boxes:
#     print(box)

In [51]:
# cap = cv2.VideoCapture("data/demo-shuttle-run_HwmEdnDL.mp4")
# cv2.namedWindow("Cone Detection", cv2.WINDOW_NORMAL)

# frame_idx = 0
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Perform detection
#     # results = model(frame, verbose=False)
    
#     break
#     # Draw detections
#     for r in results:
#         for box in r.boxes:
#             cls = int(box.cls[0])
#             label = model.names[cls]

#             # Filter ONLY cones (class name == "traffic cone" or similar)
#             if label.lower() in ["traffic cone", "cone", "ball"]:
#                 x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
#                 conf = float(box.conf[0])

#                 cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 140, 255), 2)
#                 cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 140, 255), 2)

#     cv2.imshow("Cone Detection", frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()
